# 🏦 Loan Default Prediction
**Group:** MachineStillLearning  
**Dataset:** Give Me Some Credit (Kaggle)  
**Target:** `SeriousDlqin2yrs` — binary flag for 90-day delinquency within 2 years

---
### Pipeline Overview
1. Data Loading & Memory Optimization
2. Exploratory Data Analysis
3. Preprocessing (Imputation, Outlier Removal, Scaling)
4. Feature Engineering & Resampling
5. Model Training (Logistic Regression, Random Forest, Histogram Gradient Boosting)
6. Evaluation & Comparison

## 1. Imports & Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import RobustScaler
from sklearn.ensemble import (
    RandomForestClassifier,
    HistGradientBoostingClassifier,
    IsolationForest
)
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    roc_curve,
    ConfusionMatrixDisplay
)
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler

import joblib
import os

# plotting style
sns.set_theme(style='darkgrid', palette='muted')
plt.rcParams.update({'figure.dpi': 120, 'figure.figsize': (10, 6)})

SEED = 42
np.random.seed(SEED)
print('All libraries imported successfully.')

## 2. Load Data

In [ ]:
TRAIN_PATH = '../data/raw/cs-training.csv'
TEST_PATH  = '../data/raw/cs-test.csv'

# Load — drop the unnamed index column Kaggle adds
df_train = pd.read_csv(TRAIN_PATH, index_col=0)
df_test  = pd.read_csv(TEST_PATH,  index_col=0)

print(f'Training shape : {df_train.shape}')
print(f'Test shape     : {df_test.shape}')
df_train.head()

## 3. Memory Optimization
Down-casting numerical types reduces dataset size by **~70%**, making training faster.

In [ ]:
def optimize_memory(df: pd.DataFrame) -> pd.DataFrame:
    """Down-cast int and float columns to the smallest sufficient dtype."""
    before = df.memory_usage(deep=True).sum() / 1024**2
    for col in df.select_dtypes(include=['int64', 'int32']).columns:
        df[col] = pd.to_numeric(df[col], downcast='integer')
    for col in df.select_dtypes(include=['float64', 'float32']).columns:
        df[col] = pd.to_numeric(df[col], downcast='float')
    after = df.memory_usage(deep=True).sum() / 1024**2
    print(f'Memory: {before:.2f} MB  →  {after:.2f} MB  (reduction: {100*(1 - after/before):.1f}%)')
    return df

df_train = optimize_memory(df_train)
df_test  = optimize_memory(df_test)

## 4. Exploratory Data Analysis

### 4.1 Class Distribution

In [ ]:
target = 'SeriousDlqin2yrs'
counts = df_train[target].value_counts()
pct    = df_train[target].value_counts(normalize=True) * 100

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar
bars = axes[0].bar(['Non-Default (0)', 'Default (1)'], counts.values,
                   color=['#4C72B0', '#DD8452'], edgecolor='white', linewidth=0.8)
for bar, cnt in zip(bars, counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 500,
                 f'{cnt:,}', ha='center', va='bottom', fontsize=11, fontweight='bold')
axes[0].set_title('Class Count', fontsize=13)
axes[0].set_ylabel('Count')

# Pie
axes[1].pie(counts.values, labels=['Non-Default', 'Default'],
            autopct='%1.1f%%', colors=['#4C72B0', '#DD8452'],
            startangle=90, wedgeprops={'edgecolor': 'white'})
axes[1].set_title('Class Proportion', fontsize=13)

plt.suptitle('Target Variable Distribution — SeriousDlqin2yrs', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()
print(f'Default rate: {pct[1]:.2f}%')

### 4.2 Missing Values

In [ ]:
missing = df_train.isnull().sum()
missing_pct = (df_train.isnull().sum() / len(df_train) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
missing_df = missing_df[missing_df['Missing Count'] > 0].sort_values('Missing %', ascending=False)

print('Features with missing values:')
print(missing_df)

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.barh(missing_df.index, missing_df['Missing %'], color='#c0392b', edgecolor='white')
for bar, val in zip(bars, missing_df['Missing %']):
    ax.text(val + 0.2, bar.get_y() + bar.get_height()/2, f'{val}%', va='center', fontsize=10)
ax.set_xlabel('Missing %')
ax.set_title('Missing Value Percentage by Feature', fontsize=13)
plt.tight_layout()
plt.show()

### 4.3 Correlation Heatmap

In [ ]:
plt.figure(figsize=(11, 8))
corr = df_train.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
    center=0, vmin=-1, vmax=1, linewidths=0.5,
    annot_kws={'size': 7}
)
plt.title('Feature Correlation Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()
print('Max off-diagonal absolute correlation:', corr.abs().where(~np.eye(len(corr), dtype=bool)).max().max().round(3))

### 4.4 Feature Distributions

In [ ]:
features = [c for c in df_train.columns if c != target]
n_cols = 3
n_rows = (len(features) + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, n_rows * 3.5))
axes = axes.flatten()

for i, feat in enumerate(features):
    for cls, color, label in [(0, '#4C72B0', 'Non-Default'), (1, '#DD8452', 'Default')]:
        subset = df_train[df_train[target] == cls][feat].dropna()
        # clip extreme percentiles for visibility
        clip_val = subset.quantile(0.99)
        subset = subset.clip(upper=clip_val)
        axes[i].hist(subset, bins=40, alpha=0.6, color=color, label=label, density=True)
    axes[i].set_title(feat, fontsize=9)
    axes[i].legend(fontsize=7)
    axes[i].tick_params(labelsize=7)

for j in range(i+1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Feature Distributions by Class', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 5. Preprocessing

### 5.1 Median Imputation

In [ ]:
# Compute medians on training set and apply to both splits
medians = df_train.median()
df_train.fillna(medians, inplace=True)
df_test.fillna(medians, inplace=True)

print('Missing values after imputation:')
print(df_train.isnull().sum().sum(), 'in train |', df_test.isnull().sum().sum(), 'in test')

### 5.2 Outlier Removal — Isolation Forest

In [ ]:
X_train_full = df_train.drop(columns=[target])
y_train_full = df_train[target]

iso = IsolationForest(contamination=0.008, random_state=SEED, n_jobs=-1)
outlier_labels = iso.fit_predict(X_train_full)
mask_keep = outlier_labels == 1

removed = (~mask_keep).sum()
print(f'Rows removed as outliers: {removed} ({removed/len(df_train)*100:.2f}%)')

X_clean = X_train_full[mask_keep].reset_index(drop=True)
y_clean = y_train_full[mask_keep].reset_index(drop=True)

print(f'Cleaned dataset shape: {X_clean.shape}')

### 5.3 Train / Validation Split (Stratified 80/20)

In [ ]:
X_tr, X_val, y_tr, y_val = train_test_split(
    X_clean, y_clean, test_size=0.2, random_state=SEED, stratify=y_clean
)

print(f'Train size   : {X_tr.shape[0]:,}  |  default rate: {y_tr.mean()*100:.2f}%')
print(f'Val size     : {X_val.shape[0]:,}  |  default rate: {y_val.mean()*100:.2f}%')

### 5.4 Scaling — RobustScaler

In [ ]:
scaler = RobustScaler()
X_tr_scaled  = scaler.fit_transform(X_tr)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(df_test)
print('Scaling complete. Shape:', X_tr_scaled.shape)

## 6. Feature Engineering

In [ ]:
def add_clipped_features(df: pd.DataFrame) -> pd.DataFrame:
    """Add outlier-clipped variants of highly skewed features."""
    df = df.copy()
    skewed_cols = [
        'RevolvingUtilizationOfUnsecuredLines',
        'DebtRatio',
        'NumberOfTime30-59DaysPastDueNotWorse',
        'NumberOfTimes90DaysLate',
        'NumberOfTime60-89DaysPastDueNotWorse',
    ]
    for col in skewed_cols:
        if col in df.columns:
            p99 = df[col].quantile(0.99)
            df[f'{col}_clipped'] = df[col].clip(upper=p99)
    return df

X_tr_eng  = add_clipped_features(X_tr)
X_val_eng = add_clipped_features(X_val)

print('Original features :', X_tr.shape[1])
print('After engineering  :', X_tr_eng.shape[1])

## 7. Class Imbalance Handling

In [ ]:
# --- Random Undersampling ---
rus = RandomUnderSampler(random_state=SEED)
X_tr_rus, y_tr_rus = rus.fit_resample(X_tr_scaled, y_tr)
print('After Random Undersampling:', X_tr_rus.shape, '| Default rate:', y_tr_rus.mean().round(3))

# --- SMOTE ---
smote = SMOTE(random_state=SEED, k_neighbors=5, n_jobs=-1)
X_tr_smote, y_tr_smote = smote.fit_resample(X_tr_scaled, y_tr)
print('After SMOTE            :', X_tr_smote.shape, '| Default rate:', y_tr_smote.mean().round(3))

## 8. Model Training & Evaluation

In [ ]:
def evaluate_model(name, model, X_val, y_val, proba=True):
    """Print classification report + return (accuracy, auc, y_pred, y_prob)."""
    y_pred = model.predict(X_val)
    acc = accuracy_score(y_val, y_pred)

    if proba:
        y_prob = model.predict_proba(X_val)[:, 1]
        auc = roc_auc_score(y_val, y_prob)
    else:
        y_prob = None
        auc = None

    print(f'\n{"="*55}')
    print(f'  {name}')
    print(f'{"="*55}')
    print(f'  Accuracy : {acc:.4f}')
    if auc:
        print(f'  ROC-AUC  : {auc:.4f}')
    print()
    print(classification_report(y_val, y_pred, target_names=['Non-Default', 'Default']))
    
    cm = confusion_matrix(y_val, y_pred)
    tn, fp, fn, tp = cm.ravel()
    print(f'  TN={tn:,}  FP={fp:,}  FN={fn:,}  TP={tp:,}')

    return acc, auc, y_pred, y_prob

results = {}

### 8.1 Logistic Regression (Baseline)

In [ ]:
lr = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=SEED, n_jobs=-1)
lr.fit(X_tr_scaled, y_tr)

acc, auc, y_pred_lr, y_prob_lr = evaluate_model('Logistic Regression (Baseline)', lr, X_val_scaled, y_val)
results['Logistic Regression'] = {'accuracy': acc, 'auc': auc, 'y_prob': y_prob_lr}

### 8.2 Random Forest

In [ ]:
rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    min_samples_leaf=10,
    class_weight='balanced',
    random_state=SEED,
    n_jobs=-1
)
rf.fit(X_tr, y_tr)  # tree-based — no scaling needed

acc, auc, y_pred_rf, y_prob_rf = evaluate_model('Random Forest', rf, X_val, y_val)
results['Random Forest'] = {'accuracy': acc, 'auc': auc, 'y_prob': y_prob_rf}

### 8.3 Histogram Gradient Boosting (Advanced)

In [ ]:
hgb = HistGradientBoostingClassifier(
    max_iter=300,
    learning_rate=0.05,
    max_depth=6,
    min_samples_leaf=20,
    l2_regularization=1.0,
    class_weight='balanced',
    random_state=SEED,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=20,
    verbose=0
)
hgb.fit(X_tr, y_tr)

acc, auc, y_pred_hgb, y_prob_hgb = evaluate_model('Histogram Gradient Boosting', hgb, X_val, y_val)
results['Hist Gradient Boosting'] = {'accuracy': acc, 'auc': auc, 'y_prob': y_prob_hgb}

## 9. Results Comparison

### 9.1 Summary Table

In [ ]:
summary = pd.DataFrame({
    'Model': list(results.keys()),
    'Accuracy': [results[m]['accuracy'] for m in results],
    'ROC-AUC':  [results[m]['auc']      for m in results],
}).set_index('Model').round(4)

print('\n=== Model Performance Summary ===')
print(summary.to_string())
summary

### 9.2 ROC Curve Comparison

In [ ]:
plt.figure(figsize=(9, 7))

colors = ['#4C72B0', '#55A868', '#DD8452']
probs  = [y_prob_lr, y_prob_rf, y_prob_hgb]
names  = ['Logistic Regression', 'Random Forest', 'Hist Gradient Boosting']

for prob, name, color in zip(probs, names, colors):
    fpr, tpr, _ = roc_curve(y_val, prob)
    auc_score = roc_auc_score(y_val, prob)
    plt.plot(fpr, tpr, label=f'{name} (AUC = {auc_score:.3f})', color=color, lw=2)

plt.plot([0, 1], [0, 1], 'k--', lw=1, label='Random Classifier')
plt.xlim([0, 1])
plt.ylim([0, 1.02])
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('ROC Curve Comparison', fontsize=14, fontweight='bold')
plt.legend(loc='lower right', fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 9.3 Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

model_data = [
    ('Logistic Regression', y_pred_lr),
    ('Random Forest',       y_pred_rf),
    ('Hist Gradient Boosting', y_pred_hgb),
]

for ax, (name, y_pred) in zip(axes, model_data):
    cm = confusion_matrix(y_val, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Non-Default', 'Default'])
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(name, fontsize=11, fontweight='bold')
    ax.set_xlabel('Predicted', fontsize=9)
    ax.set_ylabel('Actual', fontsize=9)

plt.suptitle('Confusion Matrices', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### 9.4 Accuracy & AUC Bar Chart

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
model_names = list(results.keys())
accuracies  = [results[m]['accuracy'] for m in model_names]
aucs        = [results[m]['auc']      for m in model_names]
bar_colors  = ['#4C72B0', '#55A868', '#DD8452']

for ax, values, title, ylabel, ylim in [
    (axes[0], accuracies, 'Model Accuracy', 'Accuracy', (0.75, 1.0)),
    (axes[1], aucs,       'Model ROC-AUC',  'AUC',      (0.80, 0.90)),
]:
    bars = ax.bar(model_names, values, color=bar_colors, edgecolor='white', linewidth=0.8)
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
                f'{val:.4f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
    ax.set_ylim(ylim)
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_ylabel(ylabel)
    ax.tick_params(axis='x', labelsize=9)

plt.tight_layout()
plt.show()

## 10. Feature Importance Analysis

In [ ]:
feature_names = X_tr.columns.tolist()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Random Forest importances
rf_imp = pd.Series(rf.feature_importances_, index=feature_names).sort_values(ascending=True)
rf_imp.plot(kind='barh', ax=axes[0], color='#55A868', edgecolor='white')
axes[0].set_title('Random Forest — Feature Importance', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Importance')

# HGB importances
hgb_imp = pd.Series(hgb.feature_importances_, index=feature_names).sort_values(ascending=True)
hgb_imp.plot(kind='barh', ax=axes[1], color='#DD8452', edgecolor='white')
axes[1].set_title('Hist Gradient Boosting — Feature Importance', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Importance')

plt.tight_layout()
plt.show()

## 11. SMOTE vs Undersampling — Impact on LR

In [ ]:
resampling_results = []

for label, X_res, y_res in [
    ('Original',   X_tr_scaled, y_tr),
    ('Undersample', X_tr_rus,   y_tr_rus),
    ('SMOTE',       X_tr_smote, y_tr_smote),
]:
    model = LogisticRegression(max_iter=1000, random_state=SEED, n_jobs=-1)
    model.fit(X_res, y_res)
    y_pred = model.predict(X_val_scaled)
    y_prob = model.predict_proba(X_val_scaled)[:, 1]
    acc = accuracy_score(y_val, y_pred)
    auc = roc_auc_score(y_val, y_prob)
    cm  = confusion_matrix(y_val, y_pred)
    tn, fp, fn, tp = cm.ravel()
    resampling_results.append({'Strategy': label, 'Accuracy': acc, 'AUC': auc, 'TP': tp, 'FP': fp, 'FN': fn, 'TN': tn})

res_df = pd.DataFrame(resampling_results).set_index('Strategy').round(4)
print('\n=== Resampling Strategy Comparison (Logistic Regression) ===')
print(res_df.to_string())
res_df

## 12. Save Best Model

In [ ]:
os.makedirs('../models', exist_ok=True)

best_model = hgb  # Hist Gradient Boosting
joblib.dump(best_model, '../models/hist_gradient_boosting.joblib')
joblib.dump(scaler,     '../models/robust_scaler.joblib')

print('Models saved to ../models/')

## 13. Conclusion

| Model | Accuracy | ROC-AUC | Notes |
|---|---|---|---|
| Logistic Regression | ~0.803 | ~0.854 | Strong baseline, many FPs |
| Random Forest | ~0.816 | ~0.858 | Better FP control, richer features |
| **Hist Gradient Boosting** | **~0.938** | **~0.865** | **Best overall — lowest FPs** |

**Key takeaways:**
- Behavioral payment history (30-59d, 90d late counts, revolving utilization) dominated predictive importance
- Income was NOT the strongest predictor — behavioral signals were
- HGB reduced false positives by >90% vs. Logistic Regression
- Median imputation + Isolation Forest outlier removal + RobustScaler = robust preprocessing pipeline

**Ethical note:** Continuous fairness monitoring, equalized-odds evaluation, and regulatory compliance are essential for responsible deployment in credit risk settings.